In [1]:
import pandas as pd
import numpy as np
from glob import glob

In [2]:
filepath_pattern = 'Data/AQI_Ozone/*.csv'
file_list = sorted(glob(filepath_pattern)) 
dataframes = []
cols_to_keep = ['Date', 'Daily Max 8-hour Ozone Concentration', 'Daily AQI Value', 'Site Latitude', 'Site Longitude', 'Local Site Name']


In [3]:
for file in file_list:
    df = pd.read_csv(file)
    df = df.loc[:, df.nunique(dropna=True) > 1]
    df = df[cols_to_keep]
    df['Date'] = pd.to_datetime(df['Date'])
    dataframes.append(df)

merged_df = pd.concat(dataframes, ignore_index=True)
temp = merged_df.groupby(['Site Latitude', 'Site Longitude'])[['Local Site Name']].count()
coords_to_drop = temp[temp['Local Site Name'] == 0].index

merged_df = merged_df[~merged_df.set_index(['Site Latitude', 'Site Longitude']).index.isin(coords_to_drop)]
duplicates = merged_df.duplicated(keep='first')
merged_df = merged_df[~duplicates]
merged_df['Date'] = pd.to_datetime(merged_df['Date'])
merged_df.head();

In [4]:
merged_df = (
    merged_df.sort_values(by='Date')
    .groupby(['Site Latitude', 'Site Longitude', 'Date'], as_index=False)
    .first()
)

merged_df.head() 
#printing this for verification
san_jose_df = merged_df[merged_df['Local Site Name'] == 'San Jose - Jackson']
san_jose_df.head()

,Site Latitude,Site Longitude,Date,Daily Max 8-hour Ozone Concentration,Daily AQI Value,Local Site Name
420648,37.348497,-121.894898,2014-01-01,0.015,14,San Jose - Jackson
420649,37.348497,-121.894898,2014-01-02,0.010,9,San Jose - Jackson
420650,37.348497,-121.894898,2014-01-03,0.010,9,San Jose - Jackson
420651,37.348497,-121.894898,2014-01-04,0.015,14,San Jose - Jackson
420652,37.348497,-121.894898,2014-01-05,0.018,17,San Jose - Jackson


In [5]:
grouped = (
    merged_df.groupby(['Site Latitude', 'Site Longitude'])
    .size()
    .reset_index(name='Count')
    .sort_values('Count', ascending=False)
)
top_40_coords = grouped.head(40)[['Site Latitude', 'Site Longitude']]


merged_df_top_40 = merged_df.merge(top_40_coords, on=['Site Latitude', 'Site Longitude'])
merged_df_top_40 = merged_df_top_40.sort_values(by=['Local Site Name', 'Date'], ascending=True).reset_index(drop=True)
merged_df_top_40.head()

,Site Latitude,Site Longitude,Date,Daily Max 8-hour Ozone Concentration,Daily AQI Value,Local Site Name
0,32.842318,-116.768293,2014-01-01,0.053,49,Alpine
1,32.842318,-116.768293,2014-01-02,0.041,38,Alpine
2,32.842318,-116.768293,2014-01-03,0.048,44,Alpine
3,32.842318,-116.768293,2014-01-04,0.047,44,Alpine
4,32.842318,-116.768293,2014-01-05,0.046,43,Alpine


In [6]:
merged_df_top_40 = merged_df_top_40.rename(columns={
    'Site Latitude': 'Latitude',
    'Site Longitude': 'Longitude',
    'Daily Max 8-hour Ozone Concentration': 'Ozone',
    'Daily AQI Value': 'AQI',
    'Local Site Name': 'Name'
})

merged_df_top_40.head()

,Latitude,Longitude,Date,Ozone,AQI,Name
0,32.842318,-116.768293,2014-01-01,0.053,49,Alpine
1,32.842318,-116.768293,2014-01-02,0.041,38,Alpine
2,32.842318,-116.768293,2014-01-03,0.048,44,Alpine
3,32.842318,-116.768293,2014-01-04,0.047,44,Alpine
4,32.842318,-116.768293,2014-01-05,0.046,43,Alpine


In [7]:
merged_df_top_40.set_index('Date', inplace=True)
merged_df_top_40.to_csv('Data/top_40_cities_ozone_data.csv', index='Date')